# Unit 1: Planning and reasoning

Same three tools as Lecture 2. Same loop. **Only the way the agent thinks changes.**

The Taj Palace is full on 14 August. That is the wall both agents hit, and only one
of them walks around it.

## Setup

In [ ]:
from cse476.lanes import get_client, MODEL, describe
from cse476.planning import (
    act_only, react, plan_then_execute, reflect, compare, NoProgress,
)
from cse476.architectures import HOTELS, ROOMS

print(describe())
client = get_client()

for name in HOTELS:
    print(f"{name:14} rooms free on 2026-08-14: {ROOMS.get((name, '2026-08-14'))}")

## 1. Act first, think never

The system prompt forbids reasoning. Everything else is identical.

Watch the trace rather than the answer.

In [ ]:
GOAL = "Book me a room at the Taj Palace on 2026-08-14."

r_act = act_only(client, MODEL, GOAL, max_steps=6)
print()
print("stopped because:", r_act["stopped_because"], "after", r_act["steps"], "steps")
print(r_act["answer"])

Notice what did **not** happen: no error, no crash, no invalid call. Every line is
a real tool called with valid arguments returning a valid result.

That is what makes this failure expensive. Nothing in that log looks wrong.

## 2. Reason, then act

Only the system prompt changed.

In [ ]:
r_react = react(client, MODEL, GOAL, max_steps=6)
print()
print("stopped because:", r_react["stopped_because"], "after", r_react["steps"], "steps")
print(r_react["answer"])

In [ ]:
print("what it was thinking:")
for i, t in enumerate(r_react["thoughts"], 1):
    print(f"  {i}. {t}")

### The mechanism

The thought is appended to the transcript as an assistant message. So on the next
turn the model reads its own stated intention sitting next to the result it got.

That is what produces "the Taj is full, so try another" **without you writing that
rule anywhere**. Reasoning works because it becomes context.

## 3. Side by side

In [ ]:
print(compare({"act only": r_act, "react": r_react}))

Step counts vary between runs, because the model is choosing. The direction of
the difference is stable. The exact numbers are not.

## 4. Exit condition three

Lecture 1 gave you two ways to stop: the goal is met, and the budget ran out. We
deferred the third. Here it is.

`NoProgress` watches two signals:

- **repeat** the same tool with the same arguments, three times over
- **stuck** different calls, but every observation says the same thing

Rerun the failing agent with the detector attached.

In [ ]:
detector = NoProgress(repeat_limit=3, stuck_limit=4)

r_guarded = react(client, MODEL, GOAL, max_steps=8, detector=detector)
print()
print("stopped because:", r_guarded["stopped_because"], "after", r_guarded["steps"], "steps")
print(r_guarded["answer"])

### The harder half

A detector that fires on healthy behaviour is worse than no detector, because it
stops working agents and teaches you to ignore it.

Run a question that **should** succeed, with the same detector, and confirm it stays
quiet.

In [ ]:
detector2 = NoProgress(repeat_limit=3, stuck_limit=4)

r_ok = react(
    client, MODEL,
    "Find me any hotel with a room free on 2026-08-14, and tell me the nightly rate.",
    max_steps=8, detector=detector2,
)
print()
print("stopped because:", r_ok["stopped_because"])
print("detector verdict:", detector2.verdict())
print(r_ok["answer"])

If `stopped_because` is `goal met` and the verdict is `None`, the detector passed
the test that actually matters.

Scenario 4 in `tests/mock_run_l4.py` proves the same thing offline.

## 5. Plan, then execute

The other shape. The plan exists as text **before anything runs**, so you can log it,
show it to a user for approval, or refuse to execute it.

None of that is possible with ReAct, where the route only exists in hindsight.

In [ ]:
r_plan = plan_then_execute(
    client, MODEL,
    "Find the cheapest hotel with a room free on 2026-08-14.",
    max_steps=8,
)
print()
print(r_plan["answer"])

## 6. Reflection

One call asks what is wrong with the draft. A second call fixes it, but only if
needed, because a good draft returns `APPROVED` and costs nothing further.

In [ ]:
weak_draft = "There is a room available somewhere."

out = reflect(client, MODEL, "Find a room on 2026-08-14 and give the price", weak_draft)
print("revised:", out["revised"])
print()
print("critique:", out["critique"])
print()
print("final:", out["final"])

### The honest limit

The same model that wrote the draft is judging it. It shares the draft's blind spots,
so the errors it is **most confident about** are exactly the ones it will approve.

Self critique catches sloppiness reliably. It catches confident wrongness rarely.
Unit 5 uses independent checks instead: a different model as reviewer, a
deterministic validator in code, or checking against a retrieved source.

## Your turn

**1. Convert your Practical 2 agent to ReAct.** Record the step count before and
after on the identical question. One table, two rows.

**2. Wire in `NoProgress`.** Then construct two questions: one that makes it fire,
and one that must not. **The second is the harder half and it is where the marks
are.** Anyone can build an alarm that goes off.

**3. Argue a shape.** For your capstone idea, make the case for ReAct or for plan
then execute in three sentences. Use the decision table from the lecture, not a
preference.

**4. Optional.** Find a question where ReAct produces sound reasoning and then an
answer that does not follow from it. It happens. When you find one, bring it, because
it is the clearest possible argument for Unit 5.

In [ ]:
# your work here
